# C-MET 小规模复现：几小时完成官方模型推理闭环

这个 Notebook 与完整复现工程完全分离。默认不处理完整 MEAD、不训练 20 万 step，而是使用官方仓库自带的示例素材、情绪特征池和官方 checkpoint，生成少量情绪编辑视频并完成技术检查。

默认输出三种情绪：`happy`、`sad`、`angry`。在 A100/L4 上通常约 1～3 小时，首次依赖和权重下载占主要时间。

> 这属于小规模推理复现，不应表述为论文训练结果复现。


## 使用方法

1. 在 Colab 菜单中选择 GPU 运行时。
2. 直接运行下一格“一键小规模复现”。
3. Google Drive 授权时点击允许。
4. 等待输出 `小规模复现完成`，下方会直接显示生成视频。

默认参数无需修改。结果会永久保存到 `MyDrive/C-MET-mini/results/official_demo/`。断线后重新运行同一格，已完成视频会自动跳过。


In [ ]:
# @title ▶ 一键小规模复现（默认约 1～3 小时）
EMOTIONS = "happy,sad,angry" # @param {type:"string"}
NUM_SAMPLES = 3 # @param {type:"integer"}
RUN_NAME = "official_demo" # @param {type:"string"}

import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
from IPython.display import Video, display


def run(command, *, cwd=None):
    command = [str(value) for value in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


if NUM_SAMPLES < 1 or NUM_SAMPLES > 10:
    raise ValueError("NUM_SAMPLES 建议保持在 1～10；默认 3。")

print("== 0/6 检查 GPU ==", flush=True)
try:
    import torch
except Exception as exc:
    raise RuntimeError("无法导入 Colab 自带的 PyTorch。") from exc
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU。请在“运行时 → 更改运行时类型”中选择 GPU。")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(["nvidia-smi"], check=False)

print("== 1/6 挂载 Google Drive ==", flush=True)
DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive 没有正确挂载。")

print("== 2/6 克隆或更新 mini 项目 ==", flush=True)
PROJECT_ROOT = Path("/content/cmet-mini-repro-colab")
PROJECT_URL = "https://github.com/ChengyangHe-ux/cmet-repro-colab.git"
PROJECT_BRANCH = "mini-repro"
if not PROJECT_ROOT.exists():
    run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch", PROJECT_URL, PROJECT_ROOT])
elif not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError(f"路径已存在但不是 Git 仓库：{PROJECT_ROOT}")
else:
    run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])

print("== 3/6 安装系统依赖 ==", flush=True)
missing = [name for name in ["ffmpeg", "ffprobe", "git-lfs"] if shutil.which(name) is None]
if missing:
    run(["apt-get", "update", "-y"])
    run(["apt-get", "install", "-y", "ffmpeg", "git-lfs"])
else:
    print("系统依赖已经存在，跳过 apt。", flush=True)

print("== 4/6 安装 Python 依赖 ==", flush=True)
run(
    [
        sys.executable,
        PROJECT_ROOT / "scripts" / "install_colab_dependencies.py",
        "--requirements",
        PROJECT_ROOT / "configs" / "colab_requirements.txt",
    ]
)

print("== 5/6 运行官方 checkpoint 小规模复现 ==", flush=True)
DRIVE_ROOT = MY_DRIVE / "C-MET-mini"
run(
    [
        sys.executable,
        PROJECT_ROOT / "scripts" / "run_colab_mini.py",
        "--project-root",
        PROJECT_ROOT,
        "--drive-root",
        DRIVE_ROOT,
        "--emotions",
        EMOTIONS,
        "--num-samples",
        NUM_SAMPLES,
        "--run-name",
        RUN_NAME,
    ]
)

print("== 6/6 展示结果 ==", flush=True)
OUTPUT_ROOT = DRIVE_ROOT / "results" / RUN_NAME
videos = sorted(OUTPUT_ROOT.glob("*.mp4"))
if not videos:
    raise RuntimeError(f"没有找到生成视频：{OUTPUT_ROOT}")
for video in videos:
    print("结果：", video.name)
    display(Video(str(video), embed=True, width=512))
print("完成。结果与 summary.json 已保存到：", OUTPUT_ROOT)


## 这条小规模链路验证了什么

- 官方 C-MET 代码可以在当前 Colab 环境加载；
- 官方 Audio2Lip、EDTalk 和 connector checkpoint 可以正确加载；
- 官方示例身份图、语音、姿态视频和 emotion2vec 情绪特征池可以组成完整输入；
- 模型能对同一说话视频生成多种目标情绪；
- 结果 MP4 同时包含视频流和音频流，并记录生成耗时。

它不包含重新训练，因此适合课程演示、项目可行性验证和快速检查，不适合声称“复现了论文全部训练结果”。
